# PROYECTO FINAL: VISIÓN ARTIFICIAL
## Auditoría Híbrida de Modelos de Visión ante Desafíos de Robustez en Entornos Dinámicos Reales
### Aplicación Sociorregional: Sistema de Detección y Estimación Biométrica (Tamaño) de Caninos en Vía Pública

**Resultados de Aprendizaje Abordados:**
* **R.A.1:** Comprende los fundamentos teóricos del procesamiento de imágenes (filtrado, transformación morfológica, umbralización y análisis de características).
* **R.A.2:** Implementación de algoritmos de extracción de características y modelos de deep learning para problemas del mundo real.
* **R.A.3:** Evaluación comparativa rigurosa mediante métricas objetivas (tiempo por frame, FPR, matrices de confusión, curvas de pérdida y análisis de robustez).

**Enfoque del Proyecto:**
Monitoreo urbano automatizado para censo canino y clasificación biométrica de tamaño (Pequeño, Mediano, Grande). Se implementan dos metodologías en paralelo:
1. **Enfoque Clásico:** Segmentación en espacio HSV, filtrado espacial no lineal (mediana), morfología matemática y descriptores de forma (área de Green, relación de aspecto, circularidad).
2. **Enfoque de Aprendizaje Profundo:** Detección de objetos y delimitación regional mediante YOLOv8 (Transfer Learning).

In [ ]:
# 1. INSTALACIÓN Y CONFIGURACIÓN DE ENTORNO HÍBRIDO (LOCAL / COLAB)
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q ultralytics
    BASE_DIR = '/content/drive/MyDrive/dog.v1i.yolov8'
else:
    BASE_DIR = './dog.v1i.yolov8' if os.path.exists('./dog.v1i.yolov8') else 'd:/Uni/Vision Artificial/proyecto/dog.v1i.yolov8'

DATASET_DIR = os.path.abspath(BASE_DIR)
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, 'train', 'images')
VALID_IMG_DIR = os.path.join(DATASET_DIR, 'valid', 'images')
TEST_IMG_DIR = os.path.join(DATASET_DIR, 'test', 'images')
TEST_LBL_DIR = os.path.join(DATASET_DIR, 'test', 'labels')

print(f"-> Entorno de ejecución: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"-> Directorio Dataset: {DATASET_DIR}")
if os.path.exists(TEST_IMG_DIR):
    print(f"-> Imágenes de Test disponibles: {len(os.listdir(TEST_IMG_DIR))}")
else:
    print("[AVISO] Carpeta de test no encontrada en la ruta especificada. Verifica la ruta.")

---
# MÓDULO 1: PROCESAMIENTO CLÁSICO (20% de la Rúbrica)

### Fundamentación Matemática y Teórica:
1. **Espacio de Color HSV:**
   A diferencia del espacio RGB donde la información cromática y de luminosidad está acoplada en los tres canales, en HSV se desacopla el matiz ($H \in [0, 179]$ en OpenCV) de la saturación ($S$) y el brillo ($V$). Esto permite definir máscaras de color tolerantes a sombras y cambios graduales de luz.
2. **Filtrado Espacial No Lineal (Filtro de Mediana):**
   $$y[i, j] = \text{mediana}\{x[i+k, j+l], (k,l) \in W\}$$
   A diferencia del filtro lineal Gaussiano (que suaviza y desenfoca los bordes de la silueta), el filtro de mediana elimina el ruido impulsivo ("sal y pimienta") preservando las fronteras nítidas del contorno del canino.
3. **Morfología Matemática:**
   * **Apertura ($A \circ B = (A \ominus B) \oplus B$):** Erosión seguida de dilatación para eliminar pequeños componentes conectados espurios (ruido de fondo o texturas de césped/suelo).
   * **Cierre ($A \bullet B = (A \oplus B) \ominus B$):** Dilatación seguida de erosión para rellenar oquedades internas dentro del pelaje del animal causadas por sombras o diferencias de textura.
4. **Descriptores Matemáticos de Forma y Tamaño:**
   * **Área ($A$):** Calculada mediante el Teorema de Green / Fórmula de Gauss (Shoelace):
     $$A = \frac{1}{2} \left| \sum_{i=0}^{n-1} (x_i y_{i+1} - x_{i+1} y_i) \right|$$
   * **Perímetro ($P$):** Longitud acumulada del contorno cerrado.
   * **Compacidad / Circularidad ($C$):** $C = \frac{4 \pi A}{P^2}$. Permite discriminar formas alargadas o irregulares frente a siluetas compactas.
   * **Relación de Aspecto ($AR$):** $AR = \frac{w}{h}$ del rectángulo mínimo contenedor (Bounding Box).
   * **Clasificación Biométrica de Tamaño:**
     * **Pequeño:** $A < 12,000 \text{ px}^2$
     * **Mediano:** $12,000 \le A < 35,000 \text{ px}^2$
     * **Grande:** $A \ge 35,000 \text{ px}^2$

In [ ]:
# 2. ANÁLISIS EXPLORATORIO DE COLOR E HISTOGRAMAS
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob

sample_images = glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))
sample_path = sample_images[0] if sample_images else os.path.join(TEST_IMG_DIR, "19_jpg.rf.c6e72c01fb3df846971cff15d321a16d.jpg")

img_bgr = cv2.imread(sample_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

# Descomposición de canales
fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
axs[0].imshow(img_rgb)
axs[0].set_title("1. Original RGB", fontsize=12, fontweight="bold")
axs[0].axis("off")

axs[1].imshow(hsv[:, :, 0], cmap="hsv")
axs[1].set_title("2. Canal H (Matiz [0-179])", fontsize=12, fontweight="bold")
axs[1].axis("off")

axs[2].imshow(hsv[:, :, 1], cmap="plasma")
axs[2].set_title("3. Canal S (Saturación [0-255])", fontsize=12, fontweight="bold")
axs[2].axis("off")

axs[3].imshow(hsv[:, :, 2], cmap="gray")
axs[3].set_title("4. Canal V (Luminancia [0-255])", fontsize=12, fontweight="bold")
axs[3].axis("off")
plt.tight_layout()
plt.show()

# Análisis estadístico de distribución de color (Histogramas)
fig, (ax_h, ax_s) = plt.subplots(1, 2, figsize=(16, 4))
ax_h.hist(hsv[:, :, 0].ravel(), bins=180, range=[0, 180], color="chocolate", alpha=0.75)
ax_h.set_title("Distribución del Histograma - Canal Hue (H)", fontsize=11, fontweight="bold")
ax_h.set_xlabel("Valor de Matiz (0 a 179)")
ax_h.set_ylabel("Frecuencia de Píxeles")
ax_h.axvspan(0, 35, color="gold", alpha=0.25, label="Rango Canino Cálido/Castaño (0-35)")
ax_h.legend()
ax_h.grid(axis="y", alpha=0.3)

ax_s.hist(hsv[:, :, 1].ravel(), bins=256, range=[0, 256], color="teal", alpha=0.75)
ax_s.set_title("Distribución del Histograma - Canal Saturación (S)", fontsize=11, fontweight="bold")
ax_s.set_xlabel("Saturación (0 a 255)")
ax_s.set_ylabel("Frecuencia de Píxeles")
ax_s.axvspan(30, 220, color="cyan", alpha=0.2, label="Rango de Saturación Efectiva (30-220)")
ax_s.legend()
ax_s.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3. PIPELINE MODULAR DE PROCESAMIENTO CLÁSICO Y DESCRIPTORES
import time

def procesar_frame_clasico(img_bgr, lower_hsv=np.array([0, 30, 40]), upper_hsv=np.array([35, 220, 255]), kernel_size=5, min_area_ruido=2500):
    """
    Pipeline de Visión Clásica para Detección y Estimación Biométrica de Tamaño.
    
    Retorna:
      img_annotated : Imagen BGR con Bounding Boxes y etiquetas
      mask_processed: Máscara binaria tras filtrado espacial y morfología
      detecciones   : Lista de diccionarios con descriptores matemáticos
      tiempo_ms     : Tiempo de cómputo exacto por frame en ms
    """
    t_inicio = time.perf_counter()
    
    # 1. Conversión de espacio de color
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    
    # 2. Umbralización de color (Segmentación)
    mask_raw = cv2.inRange(hsv, lower_hsv, upper_hsv)
    
    # 3. Filtrado Espacial No Lineal (Mediana para ruido impulsivo)
    mask_median = cv2.medianBlur(mask_raw, kernel_size)
    
    # 4. Transformaciones Morfológicas (Apertura para ruido y Cierre para vacíos en el pelaje)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    mask_opened = cv2.morphologyEx(mask_median, cv2.MORPH_OPEN, kernel)
    mask_processed = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    # 5. Extracción de Características (Contornos Geométricos)
    contours, _ = cv2.findContours(mask_processed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detecciones = []
    img_annotated = img_bgr.copy()
    
    for cnt in contours:
        # Descriptor 1: Área de Green
        area = cv2.contourArea(cnt)
        if area < min_area_ruido:
            continue  # Filtrar ruido de fondo
            
        # Descriptor 2: Perímetro y Circularidad
        perimetro = cv2.arcLength(cnt, closed=True)
        circularidad = (4 * np.pi * area) / (perimetro ** 2) if perimetro > 0 else 0.0
        
        # Descriptor 3: Bounding Box y Relación de Aspecto
        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = float(w) / float(h) if h > 0 else 0.0
        
        # Descriptor 4: Clasificación Biométrica de Tamaño
        if area >= 35000:
            tamano = "Grande"
            color_box = (0, 0, 255)    # Rojo
        elif area >= 12000:
            tamano = "Mediano"
            color_box = (0, 165, 255)  # Naranja
        else:
            tamano = "Pequeno"
            color_box = (0, 255, 0)    # Verde
            
        detecciones.append({
            "box": (x, y, w, h),
            "area": area,
            "perimetro": perimetro,
            "circularidad": circularidad,
            "aspect_ratio": aspect_ratio,
            "tamano": tamano
        })
        
        # Dibujar cuadro delimitador y rótulo de tamaño
        cv2.rectangle(img_annotated, (x, y), (x + w, y + h), color_box, 2)
        tag = f"{tamano} | Area:{int(area)} | AR:{aspect_ratio:.2f}"
        cv2.putText(img_annotated, tag, (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_box, 2, cv2.LINE_AA)
        
    t_fin = time.perf_counter()
    tiempo_ms = (t_fin - t_inicio) * 1000.0
    
    return img_annotated, mask_processed, detecciones, tiempo_ms

print("Pipeline de Visión Clásica configurado exitosamente.")

In [ ]:
# 4. PRUEBA VISUAL DEL PIPELINE EN IMAGEN INDIVIDUAL
img_test = cv2.imread(sample_path)
img_annotated, mask_proc, dets, t_frame = procesar_frame_clasico(img_test)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB))
axs[0].set_title("1. Imagen de Entrada", fontsize=12, fontweight="bold")
axs[0].axis("off")

axs[1].imshow(mask_proc, cmap="gray")
axs[1].set_title("2. Máscara Filtrada (Mediana + Morfología)", fontsize=12, fontweight="bold")
axs[1].axis("off")

axs[2].imshow(cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB))
axs[2].set_title(f"3. Detecciones Clásicas ({len(dets)} objs) - Latencia: {t_frame:.2f} ms", fontsize=12, fontweight="bold")
axs[2].axis("off")
plt.tight_layout()
plt.show()

print(f"-> Tiempo de cómputo en este frame: {t_frame:.2f} ms")
for idx, d in enumerate(dets, 1):
    print(f"   Objeto {idx}: Tamaño = {d['tamano']} | Área = {int(d['area'])} px² | Circularidad = {d['circularidad']:.3f} | Aspect Ratio = {d['aspect_ratio']:.2f}")

---
### Métricas Obligatorias del Módulo 1 (Tiempo de Cómputo por Frame y FPR)

A continuación se ejecuta la auditoría cuantitativa sobre el conjunto de prueba (`test/images`):
1. **Tiempo de Cómputo por Frame:** Medición estadística (Media $\mu$, Desviación Estándar $\sigma$, Percentil 95, FPS teóricos).
2. **Tasa de Falsos Positivos (FPR) en Entornos Estables:**
   Se contrasta cada detección del método clásico contra las anotaciones reales (Ground Truth de YOLOv8) mediante la métrica de Intersección sobre Unión (IoU):
   $$\text{IoU} = \frac{\text{Área}(B_{\text{clásico}} \cap B_{\text{real}})}{\text{Área}(B_{\text{clásico}} \cup B_{\text{real}})}$$
   * **Verdadero Positivo (TP):** Detección con $\text{IoU} \ge 0.25$ con un canino real.
   * **Falso Positivo (FP):** Detección espuria que no coincide con ningún canino (fondo, sombra o césped).
   * **Falso Negativo (FN):** Canino real no detectado por el método clásico.
   * **Tasa de Falsos Positivos por Frame:** $\frac{\sum \text{FP}}{N_{\text{frames}}}$

In [ ]:
# 5. BENCHMARK MASIVO: TIEMPOS DE CÓMPUTO POR FRAME Y TASA DE FALSOS POSITIVOS
def calcular_iou(boxA, boxB):
    # Formato box: (x, y, w, h)
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]
    unionArea = float(boxAArea + boxBArea - interArea)
    
    return interArea / unionArea if unionArea > 0 else 0.0

test_images = sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
print(f"-> Ejecutando benchmark sobre {len(test_images)} frames de prueba...")

tiempos_por_frame = []
total_tp = 0
total_fp = 0
total_fn = 0
total_gt = 0
tamano_distribucion = {"Pequeno": 0, "Mediano": 0, "Grande": 0}

for img_p in test_images:
    img = cv2.imread(img_p)
    if img is None:
        continue
    h_img, w_img = img.shape[:2]
    
    # Ejecución del pipeline clásico y toma de tiempos
    _, _, detecciones, t_ms = procesar_frame_clasico(img)
    tiempos_por_frame.append(t_ms)
    
    # Cargar Ground Truth correspondientes
    lbl_name = os.path.splitext(os.path.basename(img_p))[0] + ".txt"
    lbl_path = os.path.join(TEST_LBL_DIR, lbl_name)
    gt_boxes = []
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = map(float, parts[1:5])
                    bx = int((cx - bw / 2.0) * w_img)
                    by = int((cy - bh / 2.0) * h_img)
                    bw_px = int(bw * w_img)
                    bh_px = int(bh * h_img)
                    gt_boxes.append((bx, by, bw_px, bh_px))
                    
    total_gt += len(gt_boxes)
    gt_matched = [False] * len(gt_boxes)
    
    for det in detecciones:
        tamano_distribucion[det["tamano"]] += 1
        det_box = det["box"]
        matched = False
        for i, gt_box in enumerate(gt_boxes):
            if not gt_matched[i] and calcular_iou(det_box, gt_box) >= 0.25:
                matched = True
                gt_matched[i] = True
                break
        if matched:
            total_tp += 1
        else:
            total_fp += 1
            
    total_fn += gt_matched.count(False)

# Análisis Estadístico de Tiempos
t_array = np.array(tiempos_por_frame)
t_mean = np.mean(t_array)
t_std = np.std(t_array)
t_median = np.median(t_array)
t_p95 = np.percentile(t_array, 95)
fps_teoricos = 1000.0 / t_mean if t_mean > 0 else 0.0

precision_c = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
recall_c = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
f1_c = (2 * precision_c * recall_c) / (precision_c + recall_c) if (precision_c + recall_c) > 0 else 0.0
tasa_fp_frame = total_fp / len(test_images) if test_images else 0.0

print("=" * 68)
print("          AUDITORÍA CUANTITATIVA - MÓDULO 1: PROCESAMIENTO CLÁSICO")
print("=" * 68)
print(f"Frames evaluados             : {len(test_images)}")
print(f"Tiempo medio por frame (μ)   : {t_mean:.2f} ms ± {t_std:.2f} ms")
print(f"Mediana de tiempo            : {t_median:.2f} ms")
print(f"Percentil 95 (P95)           : {t_p95:.2f} ms")
print(f"Rendimiento Teórico          : {fps_teoricos:.1f} FPS")
print("-" * 68)
print(f"Caninos reales (Ground Truth): {total_gt}")
print(f"Verdaderos Positivos (TP)    : {total_tp}")
print(f"Falsos Positivos (FP)        : {total_fp}")
print(f"Falsos Negativos (FN)        : {total_fn}")
print(f"Precisión Clásica            : {precision_c * 100:.2f}%")
print(f"Sensibilidad / Recall        : {recall_c * 100:.2f}%")
print(f"F1-Score Clásico             : {f1_c:.4f}")
print(f"Tasa de Falsos Positivos     : {tasa_fp_frame:.2f} FP/frame")
print(f"Distribución por Tamaños     : {tamano_distribucion}")
print("=" * 68)

# Gráficas de Rendimiento y Latencia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4.5))
ax1.hist(t_array, bins=20, color="dodgerblue", edgecolor="black", alpha=0.75)
ax1.axvline(t_mean, color="red", linestyle="dashed", linewidth=2, label=f"Media: {t_mean:.2f} ms")
ax1.axvline(t_p95, color="orange", linestyle="dotted", linewidth=2, label=f"P95: {t_p95:.2f} ms")
ax1.set_title("Distribución del Tiempo de Cómputo por Frame", fontsize=11, fontweight="bold")
ax1.set_xlabel("Tiempo por Frame (ms)")
ax1.set_ylabel("Frecuencia")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.boxplot(t_array, vert=False, patch_artist=True, boxprops=dict(facecolor="lightcyan", color="teal"))
ax2.set_title("Diagrama de Caja (Boxplot) de Latencia Clásica", fontsize=11, fontweight="bold")
ax2.set_xlabel("Tiempo (ms)")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# MÓDULO 2: APRENDIZAJE PROFUNDO (20% de la Rúbrica)
### Entrenamiento y Transfer Learning con Red Neuronal Convolucional (YOLOv8)

Se entrena el modelo `yolov8n.pt` para resolver la detección regional del canino en paralelo con el enfoque clásico.

In [ ]:
# 6. ENTRENAMIENTO DE YOLOV8 CON TRANSFER LEARNING
from ultralytics import YOLO

# Cargar modelo preentrenado nano (Transfer Learning)
model = YOLO('yolov8n.pt')

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')

# Entrenamiento
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project=DATASET_DIR,
    name='entrenamiento_base'
)